In [14]:
import pandas as pd
from os import *
import folium
import webbrowser
from folium.plugins import MarkerCluster
from folium.plugins import FastMarkerCluster

In [12]:
%pip install pandas folium

^C
Note: you may need to restart the kernel to use updated packages.


In [15]:
# chdir("C:\\Users\\aymanerifa\\downloads\\ref\\ref")
# exploitant = pd.read_csv("sup_exploitant.txt", sep = ";")
# nature = pd.read_csv("sup_nature.txt", sep = ";")
# proprio = pd.read_csv("sup_proprietaire.txt", sep = ";")
# type_antenne = pd.read_csv("sup_type_antenne.txt", sep = ";")
# print(exploitant, "\n", nature, "\n", proprio, "\n", type_antenne)

data = r"C:\Users\moi\Downloads\data\data"
ref = r"C:\Users\moi\Downloads\ref\ref"

NB_POINTS = None   # None = tous les points, ou un nombre (ex : 50000) pour alléger


def lire(chemin):
    d = pd.read_csv(chemin, sep=";", encoding="utf-8", dtype={"STA_NM_ANFR": str}, low_memory=False)
    d.columns = d.columns.str.strip().str.upper()
    return d


def joindre(s):
    return ", ".join(sorted(set(s.dropna().astype(str))))


# 1. Chargement
support = lire(data + r"\sup_support.txt")
station = lire(data + r"\sup_station.txt")
antenne = lire(data + r"\sup_antenne.txt")
emetteur = lire(data + r"\sup_emetteur.txt")
bande = lire(data + r"\sup_bande.txt")
exploitant = lire(ref + r"\sup_exploitant.txt")
nature = lire(ref + r"\sup_nature.txt")
proprio = lire(ref + r"\sup_proprietaire.txt")
type_antenne = lire(ref + r"\sup_type_antenne.txt")

# 2. Coordonnées en degrés décimaux
support["LAT"] = (support["COR_NB_DG_LAT"] + support["COR_NB_MN_LAT"] / 60 + support["COR_NB_SC_LAT"] / 3600)
support["LAT"] = support["LAT"] * support["COR_CD_NS_LAT"].map({"N": 1, "S": -1})
support["LON"] = (support["COR_NB_DG_LON"] + support["COR_NB_MN_LON"] / 60 + support["COR_NB_SC_LON"] / 3600)
support["LON"] = support["LON"] * support["COR_CD_EW_LON"].map({"E": 1, "W": -1})

# 3. Antennes : nombre et types, par support
antenne = antenne.merge(type_antenne, on="TAE_ID", how="left")
ant_par_support = antenne.groupby("SUP_ID").agg(
    NB_ANTENNES=("AER_ID", "count"),
    TYPES_ANTENNE=("TAE_LB", joindre)
).reset_index()

# 4. Technologies, par support (émetteur -> antenne -> support)
lien_ant = antenne[["STA_NM_ANFR", "AER_ID", "SUP_ID"]].drop_duplicates()
emet = emetteur.merge(lien_ant, on=["STA_NM_ANFR", "AER_ID"], how="left")
emet_par_support = (emet[["SUP_ID", "EMR_LB_SYSTEME"]].dropna().drop_duplicates()
                    .groupby("SUP_ID")
                    .agg(TECHNOLOGIES=("EMR_LB_SYSTEME", joindre))
                    .reset_index())

# 5. Bandes de fréquence, par support (bande -> émetteur -> antenne -> support)
lien_emet = emet[["STA_NM_ANFR", "EMR_ID", "SUP_ID"]].drop_duplicates()
b = bande.merge(lien_emet, on=["STA_NM_ANFR", "EMR_ID"], how="left").dropna(subset=["SUP_ID"])
b = b.dropna(subset=["BAN_NB_F_DEB", "BAN_NB_F_FIN"])
b["BANDE"] = (b["BAN_NB_F_DEB"].astype(str) + "-" + b["BAN_NB_F_FIN"].astype(str)
              + " " + b["BAN_FG_UNITE"].fillna("").astype(str))
bande_par_support = (b[["SUP_ID", "BANDE"]].drop_duplicates()
                     .groupby("SUP_ID")
                     .agg(BANDES=("BANDE", joindre))
                     .reset_index())

# 6. Fusion de tout
df = (support
      .merge(station[["STA_NM_ANFR", "ADM_ID", "DTE_EN_SERVICE"]], on="STA_NM_ANFR", how="left")
      .merge(exploitant, on="ADM_ID", how="left")
      .merge(nature, on="NAT_ID", how="left")
      .merge(proprio, on="TPO_ID", how="left")
      .merge(ant_par_support, on="SUP_ID", how="left")
      .merge(emet_par_support, on="SUP_ID", how="left")
      .merge(bande_par_support, on="SUP_ID", how="left"))

points = df.dropna(subset=["LAT", "LON"]).copy()
if NB_POINTS:
    points = points.sample(min(NB_POINTS, len(points)))

print(len(points), "points sur la carte")


# 7. Groupes d'exploitants
def groupe_exploitant(nom):
    nom = str(nom).upper()
    if "ORANGE" in nom:
        return "Orange"
    if "SFR" in nom:
        return "SFR"
    if "BOUYGUES" in nom:
        return "Bouygues"
    if "FREE" in nom:
        return "Free"
    return "Autres"


COULEURS = {"Orange": "orange", "SFR": "red", "Bouygues": "blue", "Free": "purple", "Autres": "gray"}

points["GROUPE"] = points["ADM_LB_NOM"].map(groupe_exploitant)

# Texte court pour la bulle (le texte long ralentit énormément)
points["TEXTE"] = ("<b>" + points["ADM_LB_NOM"].fillna("?").astype(str) + "</b>"
                   + "<br>" + points["NAT_LB_NOM"].fillna("?").astype(str)
                   + "<br>" + points["TECHNOLOGIES"].fillna("?").astype(str))

# 8. Carte
carte = folium.Map(
    location=[46.6, 2.5],
    zoom_start=6,
    prefer_canvas=True,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

callback = """function (row) {
    var m = L.circleMarker(new L.LatLng(row[0], row[1]),
                           {radius: 4, color: row[2], fillColor: row[2], fillOpacity: 0.7, weight: 1});
    m.bindPopup(row[3]);
    return m;
};"""

for nom, couleur_groupe in COULEURS.items():
    sous = points[points["GROUPE"] == nom]
    calque = folium.FeatureGroup(name=nom + " (" + str(len(sous)) + ")").add_to(carte)
    donnees = [[r.LAT, r.LON, couleur_groupe, r.TEXTE] for r in sous.itertuples()]
    FastMarkerCluster(donnees, callback=callback).add_to(calque)

folium.LayerControl().add_to(carte)

carte.save("carte.html")
webbrowser.open("carte.html")

199290 points sur la carte


True

In [24]:
chdir("C:\\Users\\alexavi1\\Desktop\\ref")
exploitant = pd.read_csv("sup_exploitant.txt", sep = ";")
nature = pd.read_csv("sup_nature.txt", sep = ";")
proprio = pd.read_csv("sup_proprietaire.txt", sep = ";")
type_antenne = pd.read_csv("sup_type_antenne.txt", sep = ";")
print(exploitant, "\n", nature, "\n", proprio, "\n", type_antenne)

     ADM_ID               ADM_LB_NOM
0       149           AFRIPA TELECOM
1         2             RESEAU PRIVE
2         3              ANFR / DTCG
3         4                    ARCEP
4         5                  DAPESID
..      ...                      ...
184     285                     ZEOP
185     291  Haute Garonne Numérique
186     313                     SAMU
187     314         E SNCF SOLUTIONS
188     315                      MCE

[189 rows x 2 columns] 
        NAT_ID                                         NAT_LB_NOM
0           0                                        Sans nature
1          40                                          Sémaphore
2          41                                              Phare
3           4                          Château d'eau - réservoir
4          38                                           Immeuble
5          39                                    Local technique
6          42                                                Mât
7         

In [26]:
chdir("..\\data")
print(listdir("."))

['.DS_Store', 'sup_antenne.txt', 'sup_bande.txt', 'sup_emetteur.txt', 'sup_station.txt', 'sup_support.txt']


In [32]:
antenne = pd.read_csv("sup_antenne.txt", sep = ";")
bande = pd.read_csv("sup_bande.txt", sep = ";")
emetteur = pd.read_csv("sup_emetteur.txt", sep = ";")
station = pd.read_csv("sup_station.txt", sep = ";")
support = pd.read_csv("sup_support.txt", sep = ";")
print(antenne, "\n", bande, "\n", emetteur, "\n", station, "\n", support)

C:\Users\alexavi1\AppData\Local\Temp\ipykernel_4080\4153386891.py:1: DtypeWarning: Columns (0: STA_NM_ANFR) have mixed types. Specify dtype option on import or set low_memory=False.
  antenne = pd.read_csv("sup_antenne.txt", sep = ";")
C:\Users\alexavi1\AppData\Local\Temp\ipykernel_4080\4153386891.py:2: DtypeWarning: Columns (0: STA_NM_ANFR) have mixed types. Specify dtype option on import or set low_memory=False.
  bande = pd.read_csv("sup_bande.txt", sep = ";")
C:\Users\alexavi1\AppData\Local\Temp\ipykernel_4080\4153386891.py:4: DtypeWarning: Columns (0: STA_NM_ANFR) have mixed types. Specify dtype option on import or set low_memory=False.
  station = pd.read_csv("sup_station.txt", sep = ";")


       STA_NM_ANFR   AER_ID  TAE_ID AER_NB_DIMENSION AER_FG_RAYON  \
0         10030350  1400327      18                1            N   
1         10030430  1570898      18                1            N   
2         10030431  1573161      21                1            D   
3         10030436  1623591      12                1            N   
4         10030443  1684628      32              0,3            N   
...            ...      ...     ...              ...          ...   
674797  9883111006  2048619      17              3,8            D   
674798  9883111006  3310076      17              3,8            D   
674799  9883111007  2046616      17              2,4            D   
674800  9883111009  3738305      17              0,3            D   
674801  9883111010  3738307      17              0,3            D   

       AER_NB_AZIMUT AER_NB_ALT_BAS   SUP_ID  
0                NaN              5   602525  
1                NaN              5   706469  
2                173          